# DATA INGESTION

In [1]:
from pathlib import Path
from typing import List
import re

import pymupdf
from langchain_core.documents import Document

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from bs4 import BeautifulSoup
from docx import Document as DocxDocument

c:\Users\boroh\ELTE\Thesis\elte_chat\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


## File Loaders

In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def load_pdf(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    loader = PyPDFLoader(str(file_path))
    docs = loader.load()

    linked_from = file_path.parent.name if "linked_pdfs" in file_path.parts else None

    normalized_docs = []
    for i, doc in enumerate(docs):
        normalized_docs.append(
            Document(
                page_content=clean_text(doc.page_content),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "pdf",
                    "page": doc.metadata.get("page", i),
                    **doc.metadata,
                    **({"linked_from": linked_from} if linked_from else {})
                }
            )
        )

    return normalized_docs


def load_pdf_pymupdf(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    pdf = pymupdf.open(str(file_path))

    linked_from = file_path.parent.name if "linked_pdfs" in file_path.parts else None

    docs = []
    for page_num, page in enumerate(pdf):
        text = page.get_text("text")
        docs.append(
            Document(
                page_content=clean_text(text),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "pdf",
                    "page": page_num,
                    **({"linked_from": linked_from} if linked_from else {})
                }
            )
        )

    pdf.close()
    return docs


def load_txt(file_path: str | Path, encoding: str = "utf-8") -> List[Document]:
    file_path = Path(file_path)
    loader = TextLoader(str(file_path), encoding=encoding)
    docs = loader.load()

    normalized_docs = []
    for doc in docs:
        normalized_docs.append(
            Document(
                page_content=clean_text(doc.page_content),
                metadata={
                    "source": str(file_path),
                    "file_name": file_path.name,
                    "file_type": "txt",
                    **doc.metadata
                }
            )
        )

    return normalized_docs


def load_html(file_path) -> List[Document]:
    import re as _re
    file_path = Path(file_path)

    with open(file_path, "r", encoding="utf-8") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()

    # Wider content-div fallback
    root = (
        soup.find("main")
        or soup.find("article")
        or soup.find("div", id=_re.compile(r"(main|content|page)", _re.I))
        or soup.find("div", class_=_re.compile(r"(main|content|page)", _re.I))
        or soup.body
        or soup
    )

    # Page title for context prefix
    title_str = soup.title.string.strip() if soup.title and soup.title.string else ""

    meta_desc = ""
    meta_tag = soup.find("meta", attrs={"name": "description"})
    if meta_tag and meta_tag.get("content", "").strip():
        meta_desc = meta_tag["content"].strip()

    current_heading: str = ""
    lines: list[str] = []

    for tag in root.find_all(["h1", "h2", "h3", "h4", "h5", "h6",
                               "p", "li", "td", "dd"]):
        text = tag.get_text(separator=" ", strip=True)
        if not text:
            continue
        if tag.name in ("h1", "h2", "h3", "h4", "h5", "h6"):
            current_heading = text
        else:
            lines.append(f"[{current_heading}] {text}" if current_heading else text)

    # Filter boilerplate
    lines = [l for l in lines if len(l.split()) >= 4]

    body = "\n".join(lines)
    parts = [p for p in [title_str, meta_desc, body] if p]
    text = "\n\n".join(parts)

    linked_from = file_path.parent.name if "linked_pdfs" in file_path.parts else None

    return [
        Document(
            page_content=clean_text(text),
            metadata={
                "source": str(file_path),
                "file_name": file_path.name,
                "file_type": "html",
                "title": title_str or None,
                **({"linked_from": linked_from} if linked_from else {})
            }
        )
    ]

In [3]:
def load_docx(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    doc = DocxDocument(str(file_path))

    # Verify the document has actual content
    if all(not p.text.strip() for p in doc.paragraphs):
        print(f"  Warning: {file_path.name} has no paragraph content, skipping")
        return []

    current_heading: str = ""
    lines: list[str] = []

    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue

        if para.style.name.startswith("Heading"):
            current_heading = text
        else:
            lines.append(f"[{current_heading}] {text}" if current_heading else text)

    # Extract tables
    for table in doc.tables:
        for row in table.rows:
            cells = " | ".join(cell.text.strip() for cell in row.cells)
            if cells.replace("|", "").strip():
                lines.append(f"[{current_heading}] {cells}" if current_heading else cells)

    linked_from = file_path.parent.name if "linked_docx" in file_path.parts else None

    return [
        Document(
            page_content=clean_text("\n".join(lines)),
            metadata={
                "source": str(file_path),
                "file_name": file_path.name,
                "file_type": "docx",
                **({"linked_from": linked_from} if linked_from else {})
            }
        )
    ]

In [4]:
def load_file(file_path: str | Path) -> List[Document]:
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        return load_pdf_pymupdf(file_path)
    elif suffix in {".txt", ".md"}:
        return load_txt(file_path)
    elif suffix in {".html", ".htm"}:
        return load_html(file_path)
    elif suffix == ".docx":
        return load_docx(file_path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

def load_directory(directory: str | Path) -> List[Document]:
    directory = Path(directory)
    all_docs = []

    for file_path in directory.rglob("*"):
        if file_path.is_file():
            try:
                docs = load_file(file_path)
                all_docs.extend(docs)
                print(f"Loaded: {file_path}")
            except Exception as e:
                print(f"Skipped {file_path}: {e}")

    return all_docs

## Chunking

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json
import hashlib
from datetime import datetime

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)


def compute_file_hash(file_path: str | Path) -> str:
    """SHA-256 of file contents."""
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        for block in iter(lambda: f.read(65536), b""):
            h.update(block)
    return h.hexdigest()


def get_relative_path(file_path: str | Path, raw_dir: str | Path) -> str:
    """Forward-slash normalized path relative to raw_dir."""
    return Path(file_path).resolve().relative_to(Path(raw_dir).resolve()).as_posix()


def load_manifest(path: str | Path) -> dict:
    """Load the manifest, or return empty dict if it does not exist."""
    path = Path(path)
    if not path.exists():
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def save_manifest(manifest: dict, path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)


def save_chunks(chunks_list: list[dict], path: str | Path) -> None:
    """Save a list of chunk dicts (already in {content, metadata} form) to JSON."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(chunks_list, f, ensure_ascii=False, indent=2)


def load_chunks(path: str | Path) -> list[dict]:
    """Load existing chunks.json, or return empty list."""
    path = Path(path)
    if not path.exists():
        return []
    with open(path, encoding="utf-8") as f:
        return json.load(f)

## Storing the processed data in json

In [6]:
MIN_CHUNK_SIZE = 100

RAW_DIR              = Path("../data/raw")
CHUNKS_PATH          = Path("../data/processed/chunks.json")
MANIFEST_PATH        = Path("../data/processed/manifest.json")
PENDING_CHANGES_PATH = Path("../data/processed/pending_changes.json")

# File extensions we ingest
SUPPORTED_EXTS = {".pdf", ".txt", ".md", ".html", ".htm", ".docx"}


def chunk_file(file_path: Path, content_hash: str, source_relative: str) -> list[dict]:
    """Load a single file, split into chunks, attach metadata, return list of chunk dicts."""
    docs = load_file(file_path)
    if not docs:
        return []

    split_docs = text_splitter.split_documents(docs)
    split_docs = [d for d in split_docs if len(d.page_content) >= MIN_CHUNK_SIZE]

    chunks_out = []
    for n, doc in enumerate(split_docs):
        meta = dict(doc.metadata)
        meta["source_relative"] = source_relative
        meta["content_hash"]    = content_hash
        meta["chunk_id"]        = f"{source_relative}::chunk_{n}"
        chunks_out.append({
            "content": doc.page_content,
            "metadata": meta,
        })
    return chunks_out


# 1. Scan disk and build current state
current_files = {}  # source_relative -> (file_path, content_hash)
for file_path in RAW_DIR.rglob("*"):
    if not file_path.is_file():
        continue
    if file_path.suffix.lower() not in SUPPORTED_EXTS:
        continue
    rel = get_relative_path(file_path, RAW_DIR)
    current_files[rel] = (file_path, compute_file_hash(file_path))

# 2. Load manifest and diff
manifest = load_manifest(MANIFEST_PATH)
is_migration = not MANIFEST_PATH.exists()
if is_migration:
    print("No manifest.json found -> migration run: all files will be (re)processed")

manifest_paths = set(manifest.keys())
disk_paths     = set(current_files.keys())

new_files     = sorted(disk_paths - manifest_paths)
deleted_files = sorted(manifest_paths - disk_paths)
updated_files = sorted(
    p for p in (disk_paths & manifest_paths)
    if manifest[p]["content_hash"] != current_files[p][1]
)
unchanged_count = len((disk_paths & manifest_paths) - set(updated_files))

print(f"Scan: {len(disk_paths)} files on disk, {len(manifest_paths)} in manifest")
print(f"  new:       {len(new_files)}")
print(f"  updated:   {len(updated_files)}")
print(f"  deleted:   {len(deleted_files)}")
print(f"  unchanged: {unchanged_count}")

# 3. Load existing chunks.json and remove entries for deleted/updated files.
#    Also drop any old-schema chunks (no source_relative) — these are from the
#    pre-incremental pipeline and will be regenerated below.
existing_chunks = load_chunks(CHUNKS_PATH)
to_remove = set(deleted_files) | set(updated_files)
kept_chunks = [
    c for c in existing_chunks
    if c["metadata"].get("source_relative")
    and c["metadata"].get("source_relative") not in to_remove
]
removed_count = len(existing_chunks) - len(kept_chunks)
if removed_count:
    print(f"Removed {removed_count} chunks from chunks.json (deleted/updated/old-schema)")

# 4. Chunk new + updated files
to_process = new_files + updated_files
new_chunks_total = []
for rel in to_process:
    file_path, content_hash = current_files[rel]
    try:
        file_chunks = chunk_file(file_path, content_hash, rel)
        new_chunks_total.extend(file_chunks)
        manifest[rel] = {
            "content_hash":   content_hash,
            "chunk_count":    len(file_chunks),
            "last_processed": datetime.now().isoformat(timespec="seconds"),
        }
        print(f"  Chunked {rel}: {len(file_chunks)} chunks")
    except Exception as e:
        print(f"  Failed {rel}: {e}")

# 5. Drop deleted files from manifest
for rel in deleted_files:
    if rel in manifest:
        del manifest[rel]

# 6. Persist updated chunks.json, manifest, pending_changes
all_chunks = kept_chunks + new_chunks_total
save_chunks(all_chunks, CHUNKS_PATH)
save_manifest(manifest, MANIFEST_PATH)

pending_changes = {
    "new":     new_files,
    "updated": updated_files,
    "deleted": deleted_files,
}
PENDING_CHANGES_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(PENDING_CHANGES_PATH, "w", encoding="utf-8") as f:
    json.dump(pending_changes, f, ensure_ascii=False, indent=2)

print(f"\nTotal chunks in chunks.json: {len(all_chunks)}")
print(f"Wrote pending_changes to {PENDING_CHANGES_PATH}")
print("Run notebook 02 next to update ChromaDB.")

Scan: 482 files on disk, 482 in manifest
  new:       0
  updated:   0
  deleted:   0
  unchanged: 482

Total chunks in chunks.json: 6112
Wrote pending_changes to ..\data\processed\pending_changes.json
Run notebook 02 next to update ChromaDB.
